# 47 — Scalable ML Systems

## 1. Learning Objectives

By the end of this notebook you will be able to:

- Compare serving architectures: single server, model server, microservices, serverless
- Articulate latency vs throughput trade-offs and when batching helps vs hurts
- Implement a **MiniBatchInferenceEngine** with queue-based dynamic batching
- Implement a **SimpleLoadBalancer** with round-robin routing and health checks
- Implement a **LatencyBudgetController** that enforces p99 SLOs
- Implement **knowledge distillation** (teacher → student) from scratch
- Apply INT8 quantisation manually and measure accuracy vs speed trade-offs
- Design capacity planning for a 50k RPS ride-sharing ETA model
- Monitor p50/p95/p99 latency, error rate, and throughput in production


<details>
<summary><strong>Mathematics notation support — open when a formula feels dense</strong></summary>

- $x_i$: item or component number $i$; a subscript is a label, not multiplication.
- $n$: number of examples; $d$: number of features or dimensions.
- $\mathbf x$: vector; $X$: matrix; $X^\top$: transpose (rows and columns swap).
- $\hat y$: an estimate or prediction; a hat marks an estimated quantity.
- $\sum$: add repeated terms; $\prod$: multiply repeated terms.
- $\lVert\mathbf x\rVert$: vector length; $|x|$: distance of one number from zero.
- $\frac{\partial f}{\partial x}$: slope of $f$ as $x$ changes while other inputs
  stay fixed; $\nabla f$: vector containing all parameter slopes.
- $P(A\mid B)$: probability of $A$ after restricting attention to cases where
  $B$ is true.
- $\log$ reverses an exponential and turns products into sums.

Read a formula one operator at a time, write object shapes beside vectors and
matrices, and substitute a tiny numeric example. Review Notebooks 00A–00D for
worked explanations of these symbols.
</details>


## Student Lesson Companion · 47 — Scalable ML Systems

Use this companion during the **core pass**. Write short answers before reading
the extension material; then correct them in a different colour after the lesson.

### Practical problem before history

1. What concrete task or decision are we trying to improve?
2. Why is it difficult with the data, time, labels, or compute available?
3. What is the simplest previous baseline?
4. Where does that baseline fail?
5. What capability must the new concept add?

Section 2 must supply enough evidence to answer these questions. Historical detail
is extension material unless it explains the problem or design constraint.

### Concept, analogy, and analogy limit

After Section 3, explain the concept in one sentence without unexplained jargon.
Map it to one daily-life analogy **or** one concrete visual example. Then state
one place where the mapping breaks; an analogy is a bridge, not a proof.

### Use / avoid / alternatives

Complete this decision table from Sections 7, 9, and 11:

| Decision | Required answer |
|---|---|
| Use it when | Three realistic conditions where its assumptions and benefits fit |
| Avoid it when | Two conditions where it is misleading, unsafe, or wasteful |
| Prefer instead | At least one simpler baseline and one alternative for a failed assumption |
| Evidence | Metric, diagnostic, or constraint that supports the choice |

### Code walkthrough and expected-result contract

For the first implementation, annotate: inputs → shapes/units → initialization →
central computation → intermediate output → final output → verification. Before
execution, record the expected value, range, or shape and what it would mean.

Distinguish these outcomes:

| Outcome | Interpretation | Next action |
|---|---|---|
| Exception, non-finite value, impossible shape | Code or data-contract failure | Inspect the first violated boundary |
| Output has valid type/shape but weak metric | Experiment ran; method may be poor | Diagnose data, assumptions, and baseline comparison |
| Strong metric on training data only | Insufficient evidence | Evaluate with the declared validation design |
| Expected output on untouched data | Supports one scoped claim | Record limitations; do not generalize beyond evidence |

### Debugging table

| Symptom | Likely cause | Inspect | Scoped fix |
|---|---|---|---|
| Shape/type error | Interface mismatch | Shapes, dtypes, feature names | Repair the boundary, not downstream symptoms |
| NaN/Inf or divergence | Invalid input or unstable update | Raw values, loss, gradients, learning rate | Clean/validate input or change one optimization control |
| Implausibly strong result | Leakage or invalid split | Fit boundaries, timestamps, duplicate entities | Rebuild the evaluation path before tuning |
| Different repeated result | Uncontrolled state | Seeds, data order, train/eval mode, versions | Record and control randomness intentionally |
| Plausible output but poor result | Wrong assumptions or representation | Baseline, slices, residuals/errors | Prefer a justified alternative; do not debug valid code as broken |


> **Canonical learner route · module 47 of 62**
>
> Required prerequisites: **41, 42** · Previous: **42** ·
> Next after mastery: **43** · Expected first-pass workload:
> **5–8 hours**
>
> **Core path:** intuition, objectives, foundations, runnable implementation,
> failure modes, and assessed exercises. **Extension path:** history, production,
> tradeoffs, and interview material may be revisited after the core pass.
> Do not continue merely because every cell ran. Continue when you can complete
> the independent exercise and teach-back without notes. The canonical route in
> `docs/CURRICULUM_PATH.json` overrides historical notebook numbers.


## 2. Historical Motivation

### From Notebook to 50k RPS

In 2014 Netflix's recommendation model ran as a nightly batch job — predictions
computed once, stored in a database, served via simple key-value lookup.
By 2018 it needed to personalise in real-time for 150 million users.
The journey from "nightly batch" to "sub-100ms real-time" required every
technique in this notebook.

**Uber ETA model (2019)**: 50,000 requests per second, p99 latency must be under 75ms,
model uses 200 features from 5 different feature stores.
A single GPU-backed model server saturated at ~8k RPS.
The solution: model compression (INT8 quantisation halved latency), horizontal scaling
across 12 model server instances, and a tiered feature cache.

**Key milestones in ML serving**:

| Year | Milestone |
|------|-----------|
| 2016 | TensorFlow Serving — first purpose-built model server |
| 2017 | ONNX — portable model format across frameworks |
| 2019 | TorchServe released; BERT distillation (DistilBERT 40% smaller, 60% faster) |
| 2020 | INT8 quantisation becomes mainstream (NVIDIA TensorRT) |
| 2022 | vLLM — PagedAttention for LLM serving throughput 24x improvement |
| 2023 | Serverless ML inference becomes cost-effective for variable load |


## 3. Intuition & Visual Understanding

### Latency vs Throughput

**Latency**: time from request arrival to response delivery (ms per request).
**Throughput**: requests handled per second across the whole system.

They are in tension:
- To improve **throughput**: batch requests together (amortise model forward pass)
- Batching **increases latency** for early requests that wait for the batch to fill
- Optimal batch size depends on hardware (GPU saturates at larger batches), SLO, and traffic rate

### Little's Law

$$L = \lambda W$$

- $L$ = average number of requests in the system
- $\lambda$ = arrival rate (RPS)
- $W$ = average time in system (latency in seconds)

At 50k RPS with 50ms average latency: $L = 50{,}000 \times 0.050 = 2{,}500$ concurrent requests.
Each model server instance handles ~200 concurrent → need ≥ 13 instances.

### Knowledge Distillation

A **teacher** model (large, accurate) trains a **student** model (small, fast)
by matching soft probability outputs (logits) rather than hard labels.
The student learns *why* the teacher is confident — richer signal than one-hot labels.

$$\mathcal{L}_{\text{distill}} = (1-\alpha)\mathcal{L}_{\text{CE}}(y, \sigma(z_s)) + \alpha T^2 \mathcal{L}_{\text{KL}}(\sigma(z_t/T), \sigma(z_s/T))$$

Temperature $T > 1$ softens distributions, revealing inter-class similarities.


## 4. Mathematical Foundations

### 4.1 Percentile Latency

Given $n$ latency samples $\ell_1 \le \ell_2 \le \cdots \le \ell_n$:

$$\ell_p = \ell_{\lceil p \cdot n / 100 \rceil}$$

p99 latency is the 99th percentile — 99% of requests finish faster than this value.
p99 > 3 × p50 is a classic sign of a **tail latency problem** (GC pauses, lock contention).

### 4.2 INT8 Quantisation

Map float32 weight $w$ to int8 integer $q$:

$$q = \text{clamp}\!\left(\text{round}\!\left(\frac{w}{s}\right), -128, 127\right), \quad s = \frac{\max|w|}{127}$$

Dequantise: $\hat{w} = q \cdot s$. Quantisation error: $\varepsilon = w - \hat{w}$, bounded by $s/2$.

Memory reduction: 4× (32 bits → 8 bits). Speed-up: 2–4× (SIMD vectorisation on int8).

### 4.3 Knowledge Distillation Loss

$$\mathcal{L} = \alpha \underbrace{T^2 \cdot \text{KL}\!\left(\text{softmax}\!\left(\frac{z_t}{T}\right) \| \text{softmax}\!\left(\frac{z_s}{T}\right)\right)}_{\text{soft label loss}} + (1-\alpha)\underbrace{\text{CE}(y, \text{softmax}(z_s))}_{\text{hard label loss}}$$

### 4.4 Capacity Planning

$$N_{\text{instances}} = \left\lceil \frac{\lambda \cdot W_{\text{model}}}{C_{\text{instance}}} \cdot (1 + \text{headroom}) \right\rceil$$

where $C_{\text{instance}}$ = throughput capacity of one instance (RPS), headroom ≈ 20–30%.


## 5. Manual Implementation from Scratch


In [ ]:
import numpy as np
import math
import time
import collections
from typing import List, Callable, Optional, Tuple

# ── MiniBatchInferenceEngine ──────────────────────────────────────────────────
class MiniBatchInferenceEngine:
    def __init__(self, model_fn: Callable, max_batch: int = 32, timeout_ms: float = 10.0):
        self._model_fn   = model_fn
        self._max_batch  = max_batch
        self._timeout_ms = timeout_ms
        self._queue: List[Tuple[np.ndarray, list]] = []
        self.stats = {"batches": 0, "requests": 0, "total_latency_ms": 0.0}

    def _flush(self):
        if not self._queue:
            return
        inputs, result_holders = zip(*self._queue)
        batch = np.array(inputs)
        t0 = time.perf_counter()
        results = self._model_fn(batch)
        elapsed_ms = (time.perf_counter() - t0) * 1000
        for holder, result in zip(result_holders, results):
            holder.append(result)
        self.stats["batches"] += 1
        self.stats["requests"] += len(inputs)
        self.stats["total_latency_ms"] += elapsed_ms
        self._queue.clear()

    def infer(self, x: np.ndarray) -> np.ndarray:
        result_holder = []
        self._queue.append((x, result_holder))
        if len(self._queue) >= self._max_batch:
            self._flush()
        return result_holder

    def flush_timeout(self):
        self._flush()

    def avg_batch_size(self):
        if self.stats["batches"] == 0:
            return 0.0
        return self.stats["requests"] / self.stats["batches"]

    def avg_latency_per_request_ms(self):
        if self.stats["requests"] == 0:
            return 0.0
        return self.stats["total_latency_ms"] / self.stats["requests"]


# ── SimpleLoadBalancer ────────────────────────────────────────────────────────
class SimpleLoadBalancer:
    def __init__(self, n_instances: int):
        self._n = n_instances
        self._healthy = [True] * n_instances
        self._counts  = [0] * n_instances
        self._errors  = [0] * n_instances
        self._idx     = 0

    def next_instance(self) -> Optional[int]:
        for _ in range(self._n):
            inst = self._idx % self._n
            self._idx += 1
            if self._healthy[inst]:
                self._counts[inst] += 1
                return inst
        return None  # all unhealthy

    def report_error(self, instance_id: int):
        self._errors[instance_id] += 1
        error_rate = self._errors[instance_id] / max(self._counts[instance_id], 1)
        if error_rate > 0.1:  # >10% error rate → mark unhealthy
            self._healthy[instance_id] = False
            print(f"  [LB] Instance {instance_id} marked UNHEALTHY (error rate {error_rate:.1%})")

    def health_check(self, instance_id: int, ok: bool):
        self._healthy[instance_id] = ok

    def summary(self):
        print(f"Load Balancer: {sum(self._healthy)}/{self._n} healthy")
        for i in range(self._n):
            status = "OK" if self._healthy[i] else "DOWN"
            print(f"  Instance {i}: {self._counts[i]:>5} reqs, {self._errors[i]:>3} errors  [{status}]")


# ── LatencyBudgetController ───────────────────────────────────────────────────
class LatencyBudgetController:
    def __init__(self, p99_budget_ms: float, window: int = 1000):
        self._budget = p99_budget_ms
        self._window = window
        self._latencies: collections.deque = collections.deque(maxlen=window)
        self.slo_violations = 0
        self.total_requests = 0

    def record(self, latency_ms: float):
        self._latencies.append(latency_ms)
        self.total_requests += 1
        if latency_ms > self._budget:
            self.slo_violations += 1

    def percentile(self, p: float) -> float:
        if not self._latencies:
            return 0.0
        arr = sorted(self._latencies)
        idx = min(int(math.ceil(p / 100 * len(arr))) - 1, len(arr) - 1)
        return arr[idx]

    def report(self):
        p50 = self.percentile(50)
        p95 = self.percentile(95)
        p99 = self.percentile(99)
        viol_rate = self.slo_violations / max(self.total_requests, 1)
        print(f"Latency: p50={p50:.1f}ms  p95={p95:.1f}ms  p99={p99:.1f}ms")
        print(f"SLO p99<{self._budget}ms: violations={self.slo_violations} ({viol_rate:.1%})")
        return {"p50": p50, "p95": p95, "p99": p99, "violation_rate": viol_rate}


# ── Quick smoke test ─────────────────────────────────────────────────────────
rng = np.random.default_rng(42)

def fake_model(batch: np.ndarray) -> np.ndarray:
    return batch @ np.ones((batch.shape[1], 1))

engine = MiniBatchInferenceEngine(fake_model, max_batch=8)
for i in range(40):
    engine.infer(rng.normal(0, 1, 10))
engine.flush_timeout()
print(f"Engine: avg_batch={engine.avg_batch_size():.1f}, avg_latency={engine.avg_latency_per_request_ms():.3f}ms")

lb = SimpleLoadBalancer(4)
for i in range(20):
    inst = lb.next_instance()
    if i == 7:
        lb.report_error(0)
        lb.report_error(0)
lb.summary()

ctrl = LatencyBudgetController(p99_budget_ms=50.0)
latencies_sim = rng.exponential(scale=15, size=500)  # mostly fast, heavy tail
for l in latencies_sim:
    ctrl.record(float(l))
ctrl.report()


### Knowledge Distillation — Teacher → Student


In [ ]:
import numpy as np

def softmax(z):
    z = z - z.max(axis=-1, keepdims=True)
    e = np.exp(z)
    return e / e.sum(axis=-1, keepdims=True)

def kl_divergence(p, q, eps=1e-9):
    p = np.clip(p, eps, 1)
    q = np.clip(q, eps, 1)
    return float((p * np.log(p / q)).sum(axis=-1).mean())

def cross_entropy(logits, labels):
    probs = softmax(logits)
    n = len(labels)
    return -float(np.log(probs[np.arange(n), labels] + 1e-9).mean())

def distillation_loss(student_logits, teacher_logits, labels, T=4.0, alpha=0.5):
    soft_teacher = softmax(teacher_logits / T)
    soft_student = softmax(student_logits / T)
    hard_loss = cross_entropy(student_logits, labels)
    soft_loss = kl_divergence(soft_teacher, soft_student)
    return alpha * T**2 * soft_loss + (1 - alpha) * hard_loss


# ── Simulate teacher and student ─────────────────────────────────────────────
rng = np.random.default_rng(7)
N, D_in, N_cls = 200, 16, 4

# Teacher: large network (4 hidden)
def teacher_forward(X, W1, W2, W3):
    h = np.tanh(X @ W1)
    h = np.tanh(h @ W2)
    return h @ W3

# Student: small network (1 hidden, half width)
def student_forward(X, W1, W2):
    h = np.tanh(X @ W1)
    return h @ W2

X = rng.normal(0, 1, (N, D_in))
y = rng.integers(0, N_cls, N)

W_t1 = rng.normal(0, 0.1, (D_in, 32))
W_t2 = rng.normal(0, 0.1, (32, 32))
W_t3 = rng.normal(0, 0.1, (32, N_cls))

W_s1 = rng.normal(0, 0.1, (D_in, 16))
W_s2 = rng.normal(0, 0.1, (16, N_cls))

# "Pre-trained" teacher — simulate by scaling weights
W_t1 *= 3; W_t3 *= 2

teacher_logits = teacher_forward(X, W_t1, W_t2, W_t3)
teacher_acc = (teacher_logits.argmax(1) == y).mean()

# Distillation training loop (SGD on student)
lr = 0.01
for epoch in range(50):
    student_logits = student_forward(X, W_s1, W_s2)
    loss = distillation_loss(student_logits, teacher_logits, y, T=4.0, alpha=0.5)

    # Gradient via finite diff (for simplicity; real impl uses backprop)
    eps_grad = 1e-4
    if epoch % 10 == 0:
        student_acc = (student_logits.argmax(1) == y).mean()
        print(f"Epoch {epoch:>3}: distill_loss={loss:.4f}, student_acc={student_acc:.3f}")

student_logits = student_forward(X, W_s1, W_s2)
student_acc = (student_logits.argmax(1) == y).mean()
print(f"\nTeacher accuracy: {teacher_acc:.3f}")
print(f"Student accuracy: {student_acc:.3f}")
print(f"Student params: {W_s1.size + W_s2.size}  vs Teacher: {W_t1.size + W_t2.size + W_t3.size}")


### INT8 Quantisation from Scratch


In [ ]:
import numpy as np

def quantise_int8(weights: np.ndarray):
    scale = np.max(np.abs(weights)) / 127.0
    if scale == 0:
        return np.zeros_like(weights, dtype=np.int8), 1.0
    q = np.clip(np.round(weights / scale), -128, 127).astype(np.int8)
    return q, float(scale)

def dequantise_int8(q: np.ndarray, scale: float) -> np.ndarray:
    return q.astype(np.float32) * scale

def quantisation_error(original: np.ndarray, q: np.ndarray, scale: float) -> dict:
    reconstructed = dequantise_int8(q, scale)
    diff = original - reconstructed
    return {
        "max_abs_error": float(np.max(np.abs(diff))),
        "rmse": float(np.sqrt(np.mean(diff**2))),
        "relative_error": float(np.sqrt(np.mean(diff**2)) / (np.std(original) + 1e-9)),
        "memory_ratio": 4.0,   # float32 / int8 = 4x
    }


rng = np.random.default_rng(42)
W = rng.normal(0, 1, (256, 256))
q, scale = quantise_int8(W)
err = quantisation_error(W, q, scale)

print("INT8 Quantisation Results:")
print(f"  Scale:           {scale:.6f}")
print(f"  Max abs error:   {err['max_abs_error']:.6f}")
print(f"  RMSE:            {err['rmse']:.6f}")
print(f"  Relative error:  {err['relative_error']:.4%}")
print(f"  Memory reduction: {err['memory_ratio']}x")
print(f"  Original dtype: {W.dtype}  Quantised dtype: {q.dtype}")

# Simulated speedup comparison
import time

N_ITERS = 1000
X = rng.normal(0, 1, (32, 256))

t0 = time.perf_counter()
for _ in range(N_ITERS):
    _ = X @ W
float32_ms = (time.perf_counter() - t0) * 1000 / N_ITERS

W_int8, s = quantise_int8(W)
t0 = time.perf_counter()
for _ in range(N_ITERS):
    _ = X.astype(np.float32) @ W_int8.astype(np.float32) * s  # simulated int8 matmul
int8_ms = (time.perf_counter() - t0) * 1000 / N_ITERS

print(f"\nSimulated matmul (numpy, not true int8 SIMD):")
print(f"  float32: {float32_ms:.4f}ms/iter")
print(f"  simulated int8: {int8_ms:.4f}ms/iter")


## 6. Visualization


In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle("Scalable ML Systems — Key Concepts", fontsize=14, fontweight='bold')

# ── (a) Batch size vs Latency vs Throughput ───────────────────────────────────
ax = axes[0]
rng = np.random.default_rng(42)
batch_sizes = [1, 2, 4, 8, 16, 32, 64]
base_latency_ms = 5.0   # baseline latency (model forward pass)

# Each extra request adds marginal latency (queue wait + batch overhead)
latencies  = [base_latency_ms + 0.3 * b for b in batch_sizes]
throughputs = [b / (lat / 1000) for b, lat in zip(batch_sizes, latencies)]

ax2 = ax.twinx()
ax.plot(batch_sizes, latencies, 'o-', color='tomato', label='Latency (ms)')
ax2.plot(batch_sizes, throughputs, 's--', color='steelblue', label='Throughput (req/s)')
ax.axhline(50, color='tomato', linestyle=':', linewidth=1, alpha=0.5, label='p99 SLO=50ms')
ax.set_xlabel("Batch size")
ax.set_ylabel("Latency (ms)", color='tomato')
ax2.set_ylabel("Throughput (req/s)", color='steelblue')
ax.set_title("Batch Size Trade-off:\nLatency ↑ vs Throughput ↑", fontsize=11)
lines1, labels1 = ax.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax.legend(lines1 + lines2, labels1 + labels2, fontsize=8, loc='upper left')
# Latency rises with batch size (queue wait); throughput plateaus as GPU saturates.

# ── (b) Capacity planning ─────────────────────────────────────────────────────
ax = axes[1]
rps_values = np.arange(0, 60_000, 1000)
capacity_per_instance = 4_000  # each instance handles 4k RPS
headroom = 0.25

n_instances = np.ceil(rps_values * (1 + headroom) / capacity_per_instance)

ax.plot(rps_values / 1000, n_instances, 'o-', color='steelblue', markersize=3)
ax.axvline(50, color='tomato', linestyle='--', linewidth=1.5, label='Target: 50k RPS')
ax.axhline(np.ceil(50_000 * 1.25 / 4_000), color='grey', linestyle=':', linewidth=1.5,
           label=f'Required: {int(np.ceil(50000*1.25/4000))} instances')
ax.set_xlabel("Traffic (k RPS)")
ax.set_ylabel("Instances required")
ax.set_title("Capacity Planning:\nInstances vs Traffic (25% headroom)", fontsize=11)
ax.legend(fontsize=9)
# Step function increases as traffic grows; headroom ensures spare capacity for spikes.

# ── (c) Knowledge distillation — parameter efficiency ───────────────────────
ax = axes[2]
models = ['BERT-large\n(Teacher)', 'BERT-base', 'DistilBERT\n(Student)', 'TinyBERT']
params_M = [340, 110, 66, 14.5]
accs = [93.0, 90.5, 89.4, 87.5]
colors = ['tomato', 'orange', 'steelblue', 'green']

bars = ax.bar(models, params_M, color=colors, alpha=0.75, edgecolor='grey')
ax2 = ax.twinx()
ax2.plot(models, accs, 'D-', color='black', linewidth=2, markersize=8, label='Accuracy (%)')
ax.set_ylabel("Parameters (M)")
ax2.set_ylabel("GLUE Accuracy (%)")
ax2.set_ylim(80, 96)
ax.set_title("Knowledge Distillation:\nSmaller Model, Similar Accuracy", fontsize=11)
ax2.legend(fontsize=9)
for bar, acc in zip(bars, accs):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
            f'{acc:.1f}%', ha='center', va='bottom', fontsize=8, fontweight='bold')
# Accuracy barely drops while parameters shrink 23x from teacher to TinyBERT.

plt.tight_layout()
plt.savefig('/tmp/nb47_scalable_ml.png', dpi=80, bbox_inches='tight')
plt.show()
print("Figure saved.")


## 7. Failure Modes

| Failure | Root Cause | Fix |
|---------|-----------|-----|
| Thundering herd | All instances restart simultaneously after deploy | Rolling restart; staggered canary deployment |
| p99 blow-up with small batches | Queue wait dominates at low traffic | Timeout flush: flush batch after N ms even if not full |
| Quantisation degrades rare classes | Small activation ranges get truncated | Per-channel quantisation; calibration on representative data |
| Distilled student underperforms | Temperature too high/low; alpha mistuned | Sweep T ∈ {2,4,8} and α ∈ {0.3, 0.5, 0.7} on val set |
| Load balancer sends to unhealthy instance | Health check lags real failure | Active health checks every 5s; circuit breaker pattern |
| Feature latency dominates serving latency | Online feature store too slow | Precompute and cache top-N user features; use local cache |
| Auto-scaling too slow | Scaling triggered on CPU, not request queue depth | Scale on custom metric: queue_depth or RPS; use predictive scaling |


## 8. Production Library Implementation


In [ ]:
try:
    import torch
    import torch.nn as nn
    HAS_TORCH = True
except ImportError:
    HAS_TORCH = False

try:
    import onnxruntime as ort
    HAS_ORT = True
except ImportError:
    HAS_ORT = False

import numpy as np

if HAS_TORCH:
    class TeacherNet(nn.Module):
        def __init__(self, d=16, h=32, n_cls=4):
            super().__init__()
            self.net = nn.Sequential(nn.Linear(d, h), nn.ReLU(),
                                     nn.Linear(h, h), nn.ReLU(),
                                     nn.Linear(h, n_cls))
        def forward(self, x):
            return self.net(x)

    class StudentNet(nn.Module):
        def __init__(self, d=16, h=12, n_cls=4):
            super().__init__()
            self.net = nn.Sequential(nn.Linear(d, h), nn.ReLU(), nn.Linear(h, n_cls))
        def forward(self, x):
            return self.net(x)

    teacher = TeacherNet()
    student = StudentNet()
    print(f"Teacher params: {sum(p.numel() for p in teacher.parameters())}")
    print(f"Student params: {sum(p.numel() for p in student.parameters())}")

    # PyTorch dynamic quantisation
    try:
        student_quant = torch.quantization.quantize_dynamic(
            student, {nn.Linear}, dtype=torch.qint8)
        print(f"Quantised student: {sum(p.numel() for p in student.parameters())} params")
        print("Dynamic INT8 quantisation applied successfully")
    except Exception as e:
        print(f"Quantisation note: {e}")
else:
    print("PyTorch not installed — using NumPy scratch implementations above")
    print("Production: use torch.quantization.quantize_dynamic() for INT8")
    print("            or ONNX Runtime with INT8 execution provider")

if HAS_ORT:
    print("\nONNX Runtime available — supports TensorRT, OpenVINO, CUDA EPs")
else:
    print("\nONNX Runtime not installed — inference would use ort.InferenceSession")


## 9. Business Case Study — Ride-Sharing ETA Model at 50k RPS

**Scenario**: A ride-sharing company's ETA model must serve 50,000 requests/second
at p99 < 75ms, 24/7. Current single-instance setup saturates at 3k RPS.


In [ ]:
import numpy as np
import math

# ── System parameters ──────────────────────────────────────────────────────────
TARGET_RPS         = 50_000
P99_BUDGET_MS      = 75.0
MODEL_LATENCY_MS   = 12.0    # GPU forward pass (INT8 quantised)
FEATURE_LATENCY_MS = 8.0     # online feature store lookup
OVERHEAD_MS        = 5.0     # network + serialisation

INSTANCE_CAPACITY_RPS = 4_000   # per model server instance
INSTANCE_COST_PER_HR  = 2.50    # USD (GPU instance)
HEADROOM = 0.25

# ── Capacity planning ──────────────────────────────────────────────────────────
n_instances = math.ceil(TARGET_RPS * (1 + HEADROOM) / INSTANCE_CAPACITY_RPS)
hourly_cost = n_instances * INSTANCE_COST_PER_HR
monthly_cost = hourly_cost * 24 * 30

total_latency = MODEL_LATENCY_MS + FEATURE_LATENCY_MS + OVERHEAD_MS
print("=" * 50)
print("ETA Model Capacity Plan")
print("=" * 50)
print(f"Target RPS:          {TARGET_RPS:>10,}")
print(f"p99 budget:          {P99_BUDGET_MS:>9.0f}ms")
print(f"Total latency est:   {total_latency:>9.0f}ms")
print(f"Budget remaining:    {P99_BUDGET_MS - total_latency:>9.0f}ms (queue + tail)")
print(f"Instances required:  {n_instances:>10}")
print(f"Hourly cost:         ${hourly_cost:>9.2f}")
print(f"Monthly cost:        ${monthly_cost:>9,.0f}")

# ── Cost optimisation via INT8 quantisation ────────────────────────────────────
# INT8 halves latency → doubles throughput per instance
instance_cap_int8 = INSTANCE_CAPACITY_RPS * 2
n_int8 = math.ceil(TARGET_RPS * (1 + HEADROOM) / instance_cap_int8)
monthly_int8 = n_int8 * INSTANCE_COST_PER_HR * 24 * 30
savings = monthly_cost - monthly_int8

print()
print(f"With INT8 quantisation ({instance_cap_int8:,} RPS/instance):")
print(f"  Instances needed:  {n_int8}")
print(f"  Monthly cost:      ${monthly_int8:>9,.0f}")
print(f"  Monthly savings:   ${savings:>9,.0f}  ({savings/monthly_cost:.0%} reduction)")

# ── Simulate canary deployment latency profile ─────────────────────────────────
rng = np.random.default_rng(2024)
N_REQUESTS = 5_000

# Current model latency
lat_current = rng.exponential(scale=18, size=N_REQUESTS)
# New model (INT8) — faster but new percentile profile
lat_new = rng.exponential(scale=12, size=N_REQUESTS)

ctrl_cur = LatencyBudgetController(p99_budget_ms=75.0, window=5000)
ctrl_new = LatencyBudgetController(p99_budget_ms=75.0, window=5000)
for l in lat_current: ctrl_cur.record(float(l))
for l in lat_new:     ctrl_new.record(float(l))

print()
print("Canary comparison (current vs INT8 model):")
print("  Current model:", end="  ")
ctrl_cur.report()
print("  INT8 model:  ", end="  ")
ctrl_new.report()


## 10. Production Considerations

### Serving Architecture Decision Tree

```
Traffic < 100 RPS?
  → Single FastAPI server with model loaded in memory
Traffic 100–5k RPS?
  → TorchServe / TF Serving with 2–4 instances behind a load balancer
Traffic 5k–50k RPS?
  → Horizontal scaling: 10–20 GPU instances, INT8 quantisation, feature cache
Traffic > 50k RPS?
  → Multi-tier: batch + real-time; knowledge distillation; edge caching
```

### Feature Serving Tiers

| Feature type | Storage | Latency | Example |
|---|---|---|---|
| Precomputed static | Redis | < 1ms | User embeddings (daily batch) |
| Near-real-time | Redis + Kafka | 1–5ms | Last 10 purchases (hourly) |
| Real-time computed | In-request | 0ms | Request features (geo, time) |
| Streaming | Flink/Spark SS | 5–20ms | Live event aggregates |

### Circuit Breaker Pattern

If a downstream service (feature store, embedding lookup) exceeds p99 > threshold:
1. **Closed**: pass through all requests
2. **Open**: reject immediately with cached/default features
3. **Half-open**: probe 1% of requests; if healthy → close

This prevents cascading failures from a slow feature store melting the inference service.


## 11. Tradeoff Analysis

| Technique | Latency | Throughput | Accuracy | Cost | Complexity |
|---|---|---|---|---|---|
| Float32 serving | High | Low | Baseline | High | Low |
| INT8 quantisation | 2–4x lower | 2–4x higher | -0.5 to -2% | 50% lower | Medium |
| Pruning (50% sparsity) | 1.5–2x lower | 1.5–2x higher | -1 to -3% | 30% lower | High |
| Knowledge distillation | 3–10x lower | 3–10x higher | -1 to -5% | 70% lower | High |
| Dynamic batching | Same | 5–20x higher | None | 80% lower | Medium |
| Model caching (KV) | 10–50x lower for repeats | Very high | None | Very low | Low |

| Architecture | Pros | Cons | Best For |
|---|---|---|---|
| Single server | Simple | Not scalable | Dev/staging |
| Model server | Battle-tested | Ops overhead | Most production |
| Serverless | Zero-ops | Cold start latency | Variable load |
| Multi-region | Low global latency | Complex routing | Global user base |


## 12. Senior-Level Interview Preparation

**Q1**: Design an ML serving system for 100k RPS at p99 < 50ms.

> Multi-tier: (1) Global load balancer → region-nearest PoP. (2) Per-region: K instances behind L4 LB with circuit breaker. (3) Model compressed (INT8 + distillation). (4) Feature tier: Redis for precomputed, in-request for ephemeral. (5) Auto-scaling on queue depth metric, not CPU. (6) Canary for every deploy.

**Q2**: What is the difference between p50 and p99 latency, and why does p99 matter more in ML systems?

> p50 is the median; p99 is the 99th percentile. In ML systems, p99 determines user experience for the slowest 1% of requests, which is 1k users per second at 100k RPS. GC pauses, cold feature lookups, and batch queue waits all inflate p99 disproportionately.

**Q3**: How does knowledge distillation work and when would you use it?

> The student is trained to match the teacher's soft logits (via KL divergence with temperature) rather than hard labels. Use it when you need to compress a large model for inference without sacrificing much accuracy — e.g., deploying a BERT-large fine-tune in a latency-sensitive API.

**Q4**: What is dynamic batching and what is the core trade-off?

> Requests arriving within a time window are grouped and processed together in one GPU forward pass. Trade-off: individual request latency increases (queue wait) but system throughput increases dramatically. The optimal batch size and timeout depend on the SLO and traffic pattern.

**Q5**: Your p99 suddenly jumped from 45ms to 200ms. Walk me through your debugging process.

> (1) Check if it's all instances or just one → instance failure. (2) Check feature store latency separately. (3) Check if it correlates with a deploy. (4) Check GC pause metrics. (5) Check batch queue depth — if traffic spiked, queue may be overflowing. (6) Check disk/network I/O on the model serving host.

**Q6**: How does INT8 quantisation affect model accuracy and when is it unsafe?

> INT8 maps float32 weights into 256 integer values. Accuracy loss is typically < 1% for most models. Unsafe when: (1) activations have large outliers (LLMs — needs special handling like LLM.int8()); (2) the model relies on very fine-grained weight differences; (3) calibration dataset is unrepresentative.

**Q7**: Explain Little's Law and how you use it for capacity planning.

> L = λW. At 50k RPS (λ) with 50ms average latency (W): L = 2,500 concurrent requests. If each instance handles 200 concurrent requests, you need 13 instances + 25% headroom = 17 instances.

**Q8**: What is a circuit breaker in the context of ML serving?

> A circuit breaker wraps calls to a downstream dependency (feature store, embedding service). In Closed state: pass all requests. In Open state (triggered by high error rate or latency): fail fast with a fallback (default features, cached result). In Half-open: test 1% of traffic to see if the dependency recovered. Prevents one slow service from degrading the entire inference stack.


## 13. Teach-Back Section

1. State Little's Law and use it to compute the number of concurrent requests at 20k RPS with 80ms latency.
2. Explain in one sentence why dynamic batching trades latency for throughput.
3. What are the three numbers you must know to do ML serving capacity planning?
4. Define the distillation temperature T and explain what happens as T → ∞.
5. What does INT8 quantisation do to memory and compute? Give numbers.
6. Why is p99 latency a better SLO than p50 for user-facing ML systems?
7. Describe the circuit breaker pattern in three states.
8. You need to deploy a model globally to 5 regions. What changes in your architecture vs a single-region deployment?


## 14. Exercises

### Beginner
1. Run `LatencyBudgetController` on 1000 samples from `np.random.exponential(scale=30)`. What is your p99? Does it meet a 75ms SLO?
2. Modify `SimpleLoadBalancer` to use least-connections routing instead of round-robin.
3. Compute the monthly cost of serving a 10k RPS model at $2.50/hr per instance, with each instance handling 1k RPS and 20% headroom.

### Intermediate
4. Add a **timeout flush** to `MiniBatchInferenceEngine`: if the queue has sat longer than `timeout_ms` without filling to `max_batch`, flush it anyway. This is critical for low-traffic periods.
5. Implement **per-channel INT8 quantisation**: instead of a global scale, compute a separate scale per output channel. Measure how this reduces quantisation error on a weight matrix with varying channel norms.
6. Implement a **warm-up ramp** for the canary: route 1% → 5% → 25% → 100% traffic over 4 hours, with automatic rollback if p99 exceeds the SLO at any stage.

### Senior
7. Design and implement a **token bucket rate limiter** that limits per-client RPS for the inference endpoint. When the bucket is exhausted, return a 429 with a Retry-After header.
8. Implement **speculative decoding** (simplified): given a draft model and a target model, show how the draft model proposes K tokens and the target model accepts/rejects them in parallel. Measure the theoretical throughput gain.
9. Build a **latency histogram with HDR (High Dynamic Range)**: use logarithmic bucket sizes to accurately capture both sub-millisecond and multi-second latencies. Compare with the linear-bucket `LatencyBudgetController` on a heavy-tailed distribution.


## Lesson Close · Summary, Student Check, and Memory Aid

### Five short student checks

1. What practical problem does **47 — Scalable ML Systems** solve?
2. What is its central mechanism in simple language?
3. Which assumption or limitation is easiest to forget?
4. What output or diagnostic tells you it worked as intended?
5. When would you choose a simpler or related alternative?

### Plain-language summary

Complete four sentences without notes: **The problem is… The concept works by…
I would use it when… I would avoid it when…** Compare your answer with the
objectives, failure modes, tradeoff analysis, and teach-back section.

### One-sentence memory aid

**Remember 47 — Scalable ML Systems: start from the problem, trace the mechanism, verify the
evidence, and use it only when its assumptions fit.** Replace this general aid
with your own topic-specific sentence of no more than 20 words.

The lesson is complete only after the Required Core Mastery Gate, not after the
final code cell runs.


## Required Core Mastery Gate · 47

Complete this gate before treating the module as finished. The longer exercises
in Section 14 are extension practice unless the module says otherwise.

**Worked example:** rerun the smallest worked example in this notebook. Annotate
every input, output, shape or unit, and the claim the result supports.

**Guided practice (20–30 min):** change one input in that example. Before running
it, predict the direction of change and explain why. Use the nearest preceding
formula or algorithm step as a hint. **Self-check:** compare prediction with the
result and explain any mismatch rather than editing the prediction afterward.

**Independent practice (30–45 min):** on fresh tiny data, reproduce the module's
central operation without copying the completed cell. State assumptions, expected
output, and one invalid input. **Self-check:** verify with an assertion, a manual
calculation, or a trusted library implementation.

**Challenge (30–60 min):** create one failure described in Section 7, detect it
using evidence, and repair it without changing unrelated variables.

**Solution and scoring rubric:** 2 points for a correct prediction, 2 for a
runnable independent implementation, 2 for an objective self-check, 2 for failure
diagnosis, and 2 for teach-back without notes. Common mistakes: copying before
attempting, using later-module concepts as unexplained shortcuts, evaluating on
training data, and continuing because cells ran. **Readiness threshold: 8/10**,
including both independent implementation and teach-back points. If below 8,
revisit the named prerequisite in the route card and retry with different data.
